In [99]:
import partialRegionBDT
import pickle
import uproot
import os
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import numpy as np
import math as mth
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
from hist import Hist
import hist
import copy
from scipy import stats
import random
from datetime import date

Function that splits data into an array of training features, an array of labels, and an array mass values. Allows the training features to be used independently with the BDT from the labels and mass values

In [100]:
def format_data(inputs,features):
   trainingFeatures = inputs.loc[:, features].values
   labels = inputs.loc[:, ['label']].values
   signalMasses = inputs.loc[:, ['massPoint']].values   
   return trainingFeatures, labels, signalMasses

In [101]:
import sys
print(sys.version_info)

sys.version_info(major=3, minor=6, micro=8, releaselevel='final', serial=0)


Function that defines the negative mean squared weighted error as a loss function

In [102]:
def neg_mean_squared_weighted_error(y_true,y_pred,weights):
    weightedSum = 0
    totalWeights = 0
    for i in range(0, len(y_true)):
        weightedSum = weightedSum + weights[i]*(y_true[i] - y_pred[i])**2
        totalWeights = totalWeights + weights[i]
    
    return (-1*weightedSum)/totalWeights

This calculates an array of HE weights for a given depth provided by the input 'sfhist'. The overall weight is found by multiplying the weights for each depth, and takes a while to calculate. Because the bin edges aren't fixed, the correct weight is found by iterating over the bins until the correct one is found rather than using a direct list access. This takes a while to calculate when iterating over every depth in every event.  

In [103]:
def getHeDepthWeight(sfhist,heEnergy):
   weights = []
   for energyIndex, energy in enumerate(heEnergy):
      lastvalue = None
      foundbin = False
      for idx, value in enumerate(sfhist.values()):
         if(sfhist.axis().edges()[idx]>energy):
            if idx==1:
               heEnergy[energyIndex]=0
            weights.append(lastvalue)
            foundbin = True
            break
         lastvalue = value
      if not foundbin:
         weights.append(lastvalue)
   return np.array(weights)

Function that calculates CSC reweight values

In [104]:
def getcscStationWeight(sfhist,cscSegmentDR):
    weights = []
    for cscIndex, cscDR in enumerate(cscSegmentDR):
      if cscDR > 2:
        cscDR = -1      
      for idx, value in enumerate(sfhist.axis().edges()):
        if(cscDR<value):
            cscWeight = sfhist.values()[idx-1]
            break
      weights.append(cscWeight)
    return np.array(weights)

In [105]:
from xgboost import cv
from sklearn.model_selection import KFold

def kFoldCrossValidation_withoutTraining(inData,variables,nFolds,inData_systWeights):
    x,y,masses = format_data(inData,variables)
    
    random.seed()
    seed = random.randrange(1, 10000, 1)
    seed = 111
    
    kf = KFold(n_splits=nFolds,shuffle=True,random_state=seed)
    array_train_input = []
    array_test_input = []
    array_test_input_systWeights = []
    array_train_lbl = []
    array_test_lbl = []
    array_train_weight = []
    array_test_weight = []
    array_masses = []
    array_dtrain = []
    array_dtest = []
    idx=0
    for train_index, test_index in kf.split(x):
        print("TRAIN:", train_index, "TEST:", test_index)       
        array_train_input.append(x[train_index])
        array_test_input.append(x[test_index])
        array_test_input_systWeights.append(inData_systWeights.iloc[test_index])
        array_train_lbl.append(y[train_index])
        array_test_lbl.append(y[test_index])        
        array_masses.append(masses[test_index])        
        #train_input, test_input, train_lbl, test_lbl = x[train_index], x[test_index], y[train_index], y[test_index]
        train_input_withLabel = np.concatenate((array_train_input[idx], array_train_lbl[idx]), axis=1)  
        test_input_withLabel = np.concatenate((array_test_input[idx], array_test_lbl[idx]), axis=1)  
        variables_with_label = copy.deepcopy(variables)
        variables_with_label.append('label')
        df_test_input_withLabel = pd.DataFrame(test_input_withLabel, columns=variables_with_label)
        #pickle.dump(df_test_input_withLabel, open("BDT_output/testingDataset"+str(idx)+"_"+str(date.today())+".pkl", "wb"))                
        signalWeights=0
        backgroundWeights=0
        for i in train_input_withLabel:
            if i[len(i)-1] == 1:
                signalWeights+=i[len(i)-2]
            elif i[len(i)-1] == 0:
                backgroundWeights+=i[len(i)-2]
        shortfeat = variables[0:len(variables)-1]

        array_train_weight.append(array_train_input[idx][:,array_train_input[idx].shape[1]-1])
        array_test_weight.append(array_test_input[idx][:,array_test_input[idx].shape[1]-1])
        array_train_input[idx] = np.delete(array_train_input[idx],array_train_input[idx].shape[1]-1,1)
        array_test_input[idx] = np.delete(array_test_input[idx],array_test_input[idx].shape[1]-1,1)            
        array_dtrain.append(xgb.DMatrix(array_train_input[idx],label=array_train_lbl[idx],feature_names=shortfeat,weight=array_train_weight[idx]))
        array_dtest.append(xgb.DMatrix(array_test_input[idx],label=array_test_lbl[idx],feature_names=shortfeat,weight=array_test_weight[idx]))
        eval_set = [(array_test_input[idx][:,0:len(variables)-1],array_test_lbl[idx])]
        idx = idx+1
        
        
    return array_dtrain, array_dtest, array_train_input, array_test_input, array_train_lbl, array_test_lbl, array_train_weight, array_test_weight, array_masses, array_test_input_systWeights

In [106]:
def combiningKFolds(aBDT,aTest,aTest_array,aTestWeight,aTestMasses,aTrain,aTrain_array,aTrainWeight,nFolds, systWeights_array):
  array_y_predsAltTrain = []
  array_labelsaltTrain = []
  array_y_predsAlt = []
  array_labelsalt = []    
  all_y_predsAlt = np.array([])
  all_labelsalt = np.array([])
  all_systWeights = pd.concat(systWeights_array)
  all_aTestWeight = np.array([])
  all_aTestMasses = np.concatenate(np.concatenate( aTestMasses, axis=0 ), axis=0)
  #allTest_events = np.array([])
  allTest_events = []
    
  for idx in range(0,nFolds):
    y_predsAlt = aBDT[idx].predict(aTest_array[idx])
    all_y_predsAlt = np.concatenate([all_y_predsAlt,y_predsAlt])
    all_labelsalt = np.concatenate([all_labelsalt,aTest[idx].get_label()])
    all_aTestWeight = np.concatenate([all_aTestWeight,aTestWeight[idx]])
    for testEvent in aTest_array[idx]:
        allTest_events.append(testEvent)
    
    array_y_predsAltTrain.append(aBDT[idx].predict(aTrain_array[idx]))
    array_labelsaltTrain.append(aTrain[idx].get_label())
    array_y_predsAlt.append(aBDT[idx].predict(aTest_array[idx]))
    array_labelsalt.append(aTest[idx].get_label())    
    
  return all_y_predsAlt, all_labelsalt, all_aTestWeight, all_aTestMasses, allTest_events, all_systWeights


In [107]:
def findCloseToFirstHits(dataframe, dr):
    counts = [0] * dataframe.shape[0]

    for station in range(1,4):
        dphi2 = (dataframe['cscDPhibyStation_0']-dataframe['cscDPhibyStation_'+str(station)])**2
        deta2 = (dataframe['cscDEtabyStation_0']-dataframe['cscDEtabyStation_'+str(station)])**2
        dr2 = dphi2+deta2
        for idx, entry in enumerate(dr2):
            if entry < dr**2 and dataframe['cscDPhibyStation_0'].iloc[idx]>-0.9:
                counts[idx] = counts[idx] + 1
    return counts

Set the features to pull from the datafile. Can try new trainings by adding/removing features from the lists, but need to make sure that 'EventWeight' is still the last one. Some of the functions are lazy, and assume that the last is event weight when removing it for the BDT training. 

In [108]:
   allTrainingFeatures = ["pt","eta","phi","staDR","staPhi","staE","staChi","cscDR","HEDepth_0","HEDepth_1","HEDepth_2","HEDepth_3","HEDepth_4","HEDepth_5","HEDepth_6","probeCharge","standaloneDEoverE","cscDRbyStation_0","cscDRbyStation_1","cscDRbyStation_2","cscDRbyStation_3","trainingWeight"]
   allfeatures_noMissingHitBool = ["pt","eta","phi","staDR","staPhi","staE","staChi","cscDR","HEDepth_0","HEDepth_1","HEDepth_2","HEDepth_3","HEDepth_4","HEDepth_5","HEDepth_6","probeCharge","standaloneDEoverE","cscDRbyStation_0","cscDRbyStation_1","cscDRbyStation_2","cscDRbyStation_3","cellEdgeDeta","cellEdgeDphi","EventWeight"]
   allfeatures = ["pt","eta","phi","staDR","staPhi","staE","staChi","cscDR","HEDepth_0","HEDepth_1","HEDepth_2","HEDepth_3","HEDepth_4","HEDepth_5","HEDepth_6","found_HEDepth_0","found_HEDepth_1","found_HEDepth_2","found_HEDepth_3","found_HEDepth_4","found_HEDepth_5","found_HEDepth_6","probeCharge","standaloneDEoverE","cscDRbyStation_0","cscDRbyStation_1","cscDRbyStation_2","cscDRbyStation_3","cellEdgeDeta","cellEdgeDphi","EventWeight","PUupWeight","PUdownWeight","IDupWeight","IDdownWeight","ISOupWeight","ISOdownWeight","TrigUpWeight","TrigDownWeight","EnBinWeight","cscDPhibyStation_0","cscDPhibyStation_1","cscDPhibyStation_2","cscDPhibyStation_3","cscDEtabyStation_0","cscDEtabyStation_1","cscDEtabyStation_2","cscDEtabyStation_3"]
   systWeightFeatures = ["standaloneDEoverE","cellEdgeDeta","cellEdgeDphi","HEDepth_0","HEDepth_1","HEDepth_2","HEDepth_3","HEDepth_4","HEDepth_5","HEDepth_6","EventWeight","PUupWeight","PUdownWeight","IDupWeight","IDdownWeight","ISOupWeight","ISOdownWeight","TrigUpWeight","TrigDownWeight","EnBinWeight"]

In [109]:
   inDir="/local/cms/user/revering/datafiles/combineInputs/histograms_23_12_18"
   mcFile = uproot.open("/local/cms/user/revering/datafiles/combineInputs/histograms_23_12_18/ZmmSim_BaseEventFiltered.root")
   falsePos = partialRegionBDT.getRocRate("/local/cms/user/revering/datafiles/combineInputs/histograms_23_12_18/ZmmSim_BaseEventFiltered.root")
   events=mcFile["demo"]["partialDisappear"]["sigVariables"]
   mcInputs = events.arrays(allfeatures, library="pd")
   print(len(mcInputs))
   mcSystWeights = events.arrays(systWeightFeatures, library="pd")
   mcLabel = []
   for entry in range(mcInputs.shape[0]):
     mcLabel.append(0)
   mcInputs['label'] = mcLabel 

322698


In [110]:
hcalSfHists = []
for depth in range(7):
   hcalSfHists.append(uproot.open("/local/cms/user/revering/dphoton/slc7/CMSSW_10_6_17_patch1/src/DarkPhoton/MuAnalyzer/jupyterstuff/hcalSFs.root:hcalSF_depth"+str(depth)))
inputs = []

In [111]:
   dataFile_SameSign = uproot.open("/local/cms/user/revering/datafiles/combineInputs/histograms_23_12_18/sameSignCR_2018Data.root")
   events_data_SameSign=dataFile_SameSign["demo"]["partialDisappear"]["sigVariables"]
   dataInputs_SameSign = events_data_SameSign.arrays(allfeatures_noMissingHitBool, library="pd")
   dataLabel_SameSign = []
   massPoint_SameSign = []
   for entry in range(dataInputs_SameSign.shape[0]):
     dataLabel_SameSign.append(0)
     massPoint_SameSign.append("NA")
   dataInputs_SameSign['label'] = dataLabel_SameSign
   dataInputs_SameSign['massPoint']=massPoint_SameSign   
   dataInputs_SameSign['staPhi']=dataInputs_SameSign['staPhi']*dataInputs_SameSign['probeCharge']
   dataInputs_SameSign['trainingWeight'] = dataInputs_SameSign['EventWeight']    

Input the high HCAL events, both data and MC. Probably not needed anymore but kept in case the data vs MC check needs to be redone.

In [112]:
def mergeSignalFiles(signalArray):
    maxWeight = 0
    for signal in signalArray:
        if maxWeight < signal['EventWeight'].sum():
            maxWeight = signal['EventWeight'].sum()
    for signal in signalArray:
        signal['trainingWeight'] = (maxWeight/signal['EventWeight'].sum())*signal['EventWeight']
    if len(signalArray) == 5:
        return pd.concat([signalArray[0],signalArray[1],signalArray[2],signalArray[3],signalArray[4]])
    else:
        return pd.concat([signalArray[0],signalArray[1],signalArray[2],signalArray[3]])        

In [113]:
   inDir="/local/cms/user/revering/datafiles/combineInputs/histograms_23_12_18/"  
   sigFiles = []
   sigFiles_ecalBrem = []
   sigFiles_newSamples = []
   sigFiles_ecalBrem_newSamples = []
   sigMasses = []
   sigMasses_ecalBrem = []
   scaleFactor = {'0p01':5,'0p025':50,'0p05':200,'0p1':300,'0p2':800,'0p4':1200,'0p6':1600,'0p8':2500,'1p0':3200}
   for filename in os.listdir(inDir):
      f = os.path.join(inDir, filename)
      if os.path.isfile(f):
          if "DBrem_map" in filename and "fixBiasing" in filename:
             apmass = filename.split("_")[1]
             for mass, sf in scaleFactor.items():
               if mass in apmass:
                 print("fixBiasing")
                 print(mass)
                 sigMasses.append(mass)
                 sigEvents=uproot.open(str(inDir)+"/"+filename+":demo/partialDisappear/sigVariables")
                 if mass in '0p01' or mass in '0p025' or mass in '0p05' or mass in '0p1':
                     sigFiles_newSamples.append(sigEvents.arrays(allfeatures, library="pd"))
                     massLabel=[]
                     massPoint = []
                     for entry in range(sigFiles_newSamples[-1].shape[0]):
                       massLabel.append(1)
                       massPoint.append(mass)
                     sigFiles_newSamples[-1]['label']=massLabel
                     sigFiles_newSamples[-1]['massPoint'] = massPoint
                 else:
                     sigFiles.append(sigEvents.arrays(allfeatures, library="pd"))
                     massLabel=[]
                     massPoint = []
                     for entry in range(sigFiles[-1].shape[0]):
                       massLabel.append(1)
                       massPoint.append(mass)
                     sigFiles[-1]['label']=massLabel
                     sigFiles[-1]['massPoint'] = massPoint
          elif "DBrem_map" in filename and "ecalBrem" in filename:
             apmass = filename.split("_")[1]
             for mass, sf in scaleFactor.items():
               if mass in apmass:
                 print(mass)
                 sigMasses_ecalBrem.append(mass)
                 sigEvents=uproot.open(str(inDir)+"/"+filename+":demo/partialDisappear/sigVariables")
                 if mass in '0p01' or mass in '0p025' or mass in '0p05' or mass in '0p1':
                     sigFiles_ecalBrem_newSamples.append(sigEvents.arrays(allfeatures, library="pd"))
                     massLabel=[]
                     massPoint = []
                     for entry in range(sigFiles_ecalBrem_newSamples[-1].shape[0]):
                       massLabel.append(1)
                       massPoint.append(mass)
                     sigFiles_ecalBrem_newSamples[-1]['label']=massLabel
                     sigFiles_ecalBrem_newSamples[-1]['massPoint'] = massPoint        
                 else:
                     sigFiles_ecalBrem.append(sigEvents.arrays(allfeatures, library="pd"))
                     massLabel=[]
                     massPoint = []
                     for entry in range(sigFiles_ecalBrem[-1].shape[0]):
                       massLabel.append(1)
                       massPoint.append(mass)
                     sigFiles_ecalBrem[-1]['label']=massLabel
                     sigFiles_ecalBrem[-1]['massPoint'] = massPoint                    
   allSignals = mergeSignalFiles([sigFiles[0],sigFiles[1],sigFiles[2],sigFiles[3],sigFiles[4]])
   allSignals_ecalBrem = mergeSignalFiles([sigFiles_ecalBrem[0],sigFiles_ecalBrem[1],sigFiles_ecalBrem[2],sigFiles_ecalBrem[3],sigFiles_ecalBrem[4]])
   allSignals_newSamples = mergeSignalFiles([sigFiles_newSamples[0],sigFiles_newSamples[1],sigFiles_newSamples[2],sigFiles_newSamples[3]])
   allSignals_ecalBrem_newSamples = mergeSignalFiles([sigFiles_ecalBrem_newSamples[0],sigFiles_ecalBrem_newSamples[1],sigFiles_ecalBrem_newSamples[2],sigFiles_ecalBrem_newSamples[3]])   

fixBiasing
0p1
fixBiasing
0p05
1p0
0p05
0p8
fixBiasing
1p0
0p2
fixBiasing
0p2
fixBiasing
0p01
0p6
fixBiasing
0p4
fixBiasing
0p025
fixBiasing
0p6
fixBiasing
0p8
0p01
0p1
0p025
0p4


In [114]:
#file_path = "BDT_Files/weightedMcInputs.pkl" 
file_path = "BDT_Files/doesNotExist.pkl" 

hcalSfHists = []
for depth in range(7):
   hcalSfHists.append(uproot.open("/local/cms/user/revering/dphoton/slc7/CMSSW_10_6_17_patch1/src/DarkPhoton/MuAnalyzer/jupyterstuff/hcalSFs.root:hcalSF_depth"+str(depth)))
inputs = []

if os.path.exists(file_path):  
    with open(file_path, "rb") as file:
        weightedMcInputs = pickle.load(file)
else:        
    weightedMcInputs = mcInputs.copy()
    reweight=getHeDepthWeight(hcalSfHists[0],weightedMcInputs['HEDepth_'+str(0)])
    for depth in range(1,7):
       reweight = reweight*getHeDepthWeight(hcalSfHists[depth],weightedMcInputs['HEDepth_'+str(depth)])
    weightedMcInputs['EventWeight']=weightedMcInputs['EventWeight']*reweight  
    weightedMcInputs['trainingWeight'] = weightedMcInputs['EventWeight']
    weightedMcInputs['PUupWeight']=weightedMcInputs['PUupWeight']*reweight  
    weightedMcInputs['PUdownWeight']=weightedMcInputs['PUdownWeight']*reweight  
    weightedMcInputs['IDupWeight']=weightedMcInputs['IDupWeight']*reweight  
    weightedMcInputs['IDdownWeight']=weightedMcInputs['IDdownWeight']*reweight  
    weightedMcInputs['ISOupWeight']=weightedMcInputs['ISOupWeight']*reweight  
    weightedMcInputs['ISOdownWeight']=weightedMcInputs['ISOdownWeight']*reweight  
    weightedMcInputs['TrigUpWeight']=weightedMcInputs['TrigUpWeight']*reweight  
    weightedMcInputs['TrigDownWeight']=weightedMcInputs['TrigDownWeight']*reweight  
    weightedMcInputs['EnBinWeight']=weightedMcInputs['EnBinWeight']*reweight  
    
    # save data to file
#    with open(file_path, "wb") as file:
#        pickle.dump(weightedMcInputs, file)

/home/krohn045/.local/lib/python3.6/site-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':


In [115]:
cscSfHists = []
for depth in range(4):
    print(depth)
    cscSfHists.append(uproot.open("/local/cms/user/revering/dphoton/slc7/CMSSW_10_6_17_patch1/src/DarkPhoton/MuAnalyzer/jupyterstuff/cscStationSFs.root:cscScaleFactor_station"+str(depth)))
print(cscSfHists[0].axis().edges())
print(cscSfHists[0].values())
cscReweight0=getcscStationWeight(cscSfHists[0],weightedMcInputs['cscDRbyStation_'+str(0)])
cscReweight1=getcscStationWeight(cscSfHists[1],weightedMcInputs['cscDRbyStation_'+str(1)])
cscReweight2=getcscStationWeight(cscSfHists[2],weightedMcInputs['cscDRbyStation_'+str(2)])
cscReweight3=getcscStationWeight(cscSfHists[3],weightedMcInputs['cscDRbyStation_'+str(3)])
weightedMcInputs['EventWeight_cscReweight']=weightedMcInputs['EventWeight']*cscReweight0*cscReweight1*cscReweight2*cscReweight3

0
1
2
3
[-1.   -0.97 -0.94 -0.91 -0.88 -0.85 -0.82 -0.79 -0.76 -0.73 -0.7  -0.67
 -0.64 -0.61 -0.58 -0.55 -0.52 -0.49 -0.46 -0.43 -0.4  -0.37 -0.34 -0.31
 -0.28 -0.25 -0.22 -0.19 -0.16 -0.13 -0.1  -0.07 -0.04 -0.01  0.02  0.17
  1.97]
[6.053152   0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.97274566 0.86921054 1.3505379 ]


In [116]:
weightedInputs = []
allSignals['EventWeight_cscReweight']=allSignals['EventWeight']
allSignals_ecalBrem['EventWeight_cscReweight']=allSignals_ecalBrem['EventWeight']
allSignals_newSamples['EventWeight_cscReweight']=allSignals_newSamples['EventWeight']
allSignals_ecalBrem_newSamples['EventWeight_cscReweight']=allSignals_ecalBrem_newSamples['EventWeight']
weightedInputs.append(pd.concat([weightedMcInputs,allSignals]))

for dataset in weightedInputs:
   dataset['staPhi']=dataset['staPhi']*dataset['probeCharge']
   print(dataset.loc[dataset['found_HEDepth_0']==0,'HEDepth_0']) 
   dataset.loc[dataset['found_HEDepth_0']==0,'HEDepth_0'] = dataset.loc[dataset['found_HEDepth_0']==0,'HEDepth_0'].replace(0,-1)
   dataset.loc[dataset['found_HEDepth_1']==0,'HEDepth_1'] = dataset.loc[dataset['found_HEDepth_1']==0,'HEDepth_1'].replace(0,-1)
   dataset.loc[dataset['found_HEDepth_2']==0,'HEDepth_2'] = dataset.loc[dataset['found_HEDepth_2']==0,'HEDepth_2'].replace(0,-1)
   dataset.loc[dataset['found_HEDepth_3']==0,'HEDepth_3'] = dataset.loc[dataset['found_HEDepth_3']==0,'HEDepth_3'].replace(0,-1)
   dataset.loc[dataset['found_HEDepth_4']==0,'HEDepth_4'] = dataset.loc[dataset['found_HEDepth_4']==0,'HEDepth_4'].replace(0,-1)
   dataset.loc[dataset['found_HEDepth_5']==0,'HEDepth_5'] = dataset.loc[dataset['found_HEDepth_5']==0,'HEDepth_5'].replace(0,-1)
   dataset.loc[dataset['found_HEDepth_6']==0,'HEDepth_6'] = dataset.loc[dataset['found_HEDepth_6']==0,'HEDepth_6'].replace(0,-1)
   print(dataset.loc[dataset['found_HEDepth_0']==0,'HEDepth_0']) 
    
allSignals_ecalBrem['staPhi']=allSignals_ecalBrem['staPhi']*allSignals_ecalBrem['probeCharge']
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_0']==0,'HEDepth_0'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_0']==0,'HEDepth_0'].replace(0,-1)
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_1']==0,'HEDepth_1'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_1']==0,'HEDepth_1'].replace(0,-1)
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_2']==0,'HEDepth_2'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_2']==0,'HEDepth_2'].replace(0,-1)
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_3']==0,'HEDepth_3'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_3']==0,'HEDepth_3'].replace(0,-1)
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_4']==0,'HEDepth_4'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_4']==0,'HEDepth_4'].replace(0,-1)
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_5']==0,'HEDepth_5'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_5']==0,'HEDepth_5'].replace(0,-1)
allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_6']==0,'HEDepth_6'] = allSignals_ecalBrem.loc[allSignals_ecalBrem['found_HEDepth_6']==0,'HEDepth_6'].replace(0,-1)    

allSignals_newSamples['staPhi']=allSignals_newSamples['staPhi']*allSignals_newSamples['probeCharge']
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_0']==0,'HEDepth_0'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_0']==0,'HEDepth_0'].replace(0,-1)
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_1']==0,'HEDepth_1'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_1']==0,'HEDepth_1'].replace(0,-1)
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_2']==0,'HEDepth_2'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_2']==0,'HEDepth_2'].replace(0,-1)
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_3']==0,'HEDepth_3'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_3']==0,'HEDepth_3'].replace(0,-1)
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_4']==0,'HEDepth_4'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_4']==0,'HEDepth_4'].replace(0,-1)
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_5']==0,'HEDepth_5'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_5']==0,'HEDepth_5'].replace(0,-1)
allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_6']==0,'HEDepth_6'] = allSignals_newSamples.loc[allSignals_newSamples['found_HEDepth_6']==0,'HEDepth_6'].replace(0,-1)    

allSignals_ecalBrem_newSamples['staPhi']=allSignals_ecalBrem_newSamples['staPhi']*allSignals_ecalBrem_newSamples['probeCharge']
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_0']==0,'HEDepth_0'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_0']==0,'HEDepth_0'].replace(0,-1)
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_1']==0,'HEDepth_1'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_1']==0,'HEDepth_1'].replace(0,-1)
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_2']==0,'HEDepth_2'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_2']==0,'HEDepth_2'].replace(0,-1)
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_3']==0,'HEDepth_3'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_3']==0,'HEDepth_3'].replace(0,-1)
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_4']==0,'HEDepth_4'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_4']==0,'HEDepth_4'].replace(0,-1)
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_5']==0,'HEDepth_5'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_5']==0,'HEDepth_5'].replace(0,-1)
allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_6']==0,'HEDepth_6'] = allSignals_ecalBrem_newSamples.loc[allSignals_ecalBrem_newSamples['found_HEDepth_6']==0,'HEDepth_6'].replace(0,-1)    


2240     0.0
6119     0.0
7550     0.0
8094     0.0
12054    0.0
        ... 
1550     0.0
1787     0.0
2171     0.0
2255     0.0
2613     0.0
Name: HEDepth_0, Length: 190, dtype: float64
2240    -1.0
6119    -1.0
7550    -1.0
8094    -1.0
12054   -1.0
        ... 
1550    -1.0
1787    -1.0
2171    -1.0
2255    -1.0
2613    -1.0
Name: HEDepth_0, Length: 190, dtype: float64


In [117]:
weightedInputs2 = weightedInputs.copy()

for idx, dataset in enumerate(weightedInputs):
  stacut = -0.6
  print(len(dataset))
  filter = ((dataset['standaloneDEoverE']<stacut) & 
                                 (((dataset['eta']>0) & ((dataset['cellEdgeDeta']<-0.16) | (dataset['cellEdgeDeta'] > 0.002))) | 
                                  ((dataset['eta']<0) & ((dataset['cellEdgeDeta']<-0.002) | (dataset['cellEdgeDeta'] > 0.016)))) & 
                                 (((abs(dataset['phi'])>1.5) & ((dataset['cellEdgeDphi']<-0.001) | (dataset['cellEdgeDphi'] > 0.004))) | 
                                  ((abs(dataset['phi'])<1.5) & ((dataset['cellEdgeDphi']<-0.004) | (dataset['cellEdgeDphi'] > 0.001)))))
                                 
  filteredData = dataset[filter]
    
  #Find the number of close hits
  dr = 0.05
  nCloseHitsColumn = findCloseToFirstHits(filteredData,dr)
  #Add them to the dataframe
  filteredData['closeHits'] = nCloseHitsColumn
  #Apply our cut
  #filteredData = filteredData[filteredData['closeHits']>1]
    
  filteredData.drop('cellEdgeDphi', inplace=True, axis=1)
  filteredData.drop('cellEdgeDeta', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_0', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_1', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_2', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_3', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_4', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_5', inplace=True, axis=1)
  filteredData.drop('found_HEDepth_6', inplace=True, axis=1)    
  filteredData.drop('cscDPhibyStation_0', inplace=True, axis=1)  
  filteredData.drop('cscDPhibyStation_1', inplace=True, axis=1)  
  filteredData.drop('cscDPhibyStation_2', inplace=True, axis=1)  
  filteredData.drop('cscDPhibyStation_3', inplace=True, axis=1)  
  filteredData.drop('cscDEtabyStation_0', inplace=True, axis=1)  
  filteredData.drop('cscDEtabyStation_1', inplace=True, axis=1)  
  filteredData.drop('cscDEtabyStation_2', inplace=True, axis=1)  
  filteredData.drop('cscDEtabyStation_3', inplace=True, axis=1)    

  missingOneHit = filteredData.copy()
    
  missingOneHit_SystWeights = missingOneHit.copy()
  missingOneHit_SystWeights.drop(['pt'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['eta'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['phi'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['staDR'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['staPhi'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['staE'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['staChi'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['cscDR'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_0'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_1'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_2'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_3'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_4'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_5'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['HEDepth_6'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['probeCharge'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['standaloneDEoverE'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['cscDRbyStation_0'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['cscDRbyStation_1'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['cscDRbyStation_2'],inplace=True, axis=1)
  missingOneHit_SystWeights.drop(['cscDRbyStation_3'],inplace=True, axis=1)       

  missingOneHit.drop(['PUupWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['PUdownWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['IDupWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['IDdownWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['ISOupWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['ISOdownWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['TrigUpWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['TrigDownWeight'],inplace=True, axis=1)    
  missingOneHit.drop(['EnBinWeight'],inplace=True, axis=1) 
  missingOneHit.drop(['EventWeight_cscReweight'],inplace=True, axis=1) 
  missingOneHit.drop('closeHits', inplace=True, axis=1)


360626


/home/krohn045/.local/lib/python3.6/site-packages/ipykernel_launcher.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


In [118]:
bdtArray = []
bdtArray.append(pickle.load(open("BDT_output/partialRegion0_2023-08-08.pickle.dat", "rb")))
bdtArray.append(pickle.load(open("BDT_output/partialRegion1_2023-08-08.pickle.dat", "rb")))
bdtArray.append(pickle.load(open("BDT_output/partialRegion2_2023-08-08.pickle.dat", "rb")))
bdtArray.append(pickle.load(open("BDT_output/partialRegion3_2023-08-08.pickle.dat", "rb")))
bdtArray.append(pickle.load(open("BDT_output/partialRegion4_2023-08-08.pickle.dat", "rb")))

In [119]:
  nFolds = 5
  importance = []
  sigPreds = []
  newGenPreds = [] 

  array_train, array_test, array_train_array, array_test_array, array_train_lbl, array_test_lbl, array_trainWeight, array_testWeight, array_testMasses, array_SystWeights = kFoldCrossValidation_withoutTraining(missingOneHit,allTrainingFeatures,nFolds,missingOneHit_SystWeights)
  print(len(bdtArray[0].evals_result()))
  print(bdtArray[0].evals_result())
  print("len(array_test_array)")
  print(len(array_test_array))

  y_preds, labels, weights, masses_labels, testEvents_array, BDToutput_systWeights = combiningKFolds(bdtArray,array_test,array_test_array,array_testWeight,array_testMasses,array_train,array_train_array,array_trainWeight,nFolds, array_SystWeights)

  print(len(BDToutput_systWeights))
  print("DEBUG")
  print(len(testEvents_array))
  print(len(y_preds))
  print(len(weights))
  print(len(labels))

  #Using predictions of testing dataset
  bkg_predsAlt = []
  sig0p2_predsAlt = []
  sig0p4_predsAlt = []
  sig0p6_predsAlt = []
  sig0p8_predsAlt = []
  sig1p0_predsAlt = []
  for idx2, BDTscore in enumerate(y_preds):
      if labels[idx2] == 0: bkg_predsAlt.append(BDTscore)
      else:
            if masses_labels[idx2] == "0p2": sig0p2_predsAlt.append(BDTscore)
            elif masses_labels[idx2] == "0p4": sig0p4_predsAlt.append(BDTscore)
            elif masses_labels[idx2] == "0p6": sig0p6_predsAlt.append(BDTscore)
            elif masses_labels[idx2] == "0p8": sig0p8_predsAlt.append(BDTscore)
            elif masses_labels[idx2] == "1p0": sig1p0_predsAlt.append(BDTscore)


  newGenPreds.append(bkg_predsAlt)

  sigPreds.append(sig1p0_predsAlt)
  sigPreds.append(sig0p2_predsAlt)    
  sigPreds.append(sig0p4_predsAlt)
  sigPreds.append(sig0p6_predsAlt)
  sigPreds.append(sig0p8_predsAlt)


TRAIN: [     1      2      3 ... 109862 109863 109864] TEST: [     0     19     24 ... 109835 109838 109846]
TRAIN: [     0      1      3 ... 109862 109863 109864] TEST: [     2     17     20 ... 109852 109853 109858]
TRAIN: [     0      1      2 ... 109862 109863 109864] TEST: [     3      7      9 ... 109856 109859 109861]
TRAIN: [     0      1      2 ... 109861 109862 109863] TEST: [     4      5      6 ... 109847 109855 109864]
TRAIN: [     0      2      3 ... 109859 109861 109864] TEST: [     1     10     12 ... 109860 109862 109863]
1
{'validation_0': OrderedDict([('logloss', [0.299802, 0.236386, 0.231063, 0.242951, 0.252297, 0.265006, 0.280849, 0.285622, 0.281733, 0.276388, 0.276169, 0.271965, 0.270234, 0.265449, 0.264462, 0.259309, 0.258668, 0.259265, 0.257529, 0.256319, 0.253896, 0.249393, 0.252044, 0.251431, 0.249196, 0.249878, 0.248893, 0.245453, 0.244138, 0.247253, 0.245409, 0.242444, 0.244388, 0.2424, 0.241272, 0.241235, 0.241331, 0.239493, 0.238335, 0.237644, 0.235434, 0.

In [120]:
  stacut = -0.6
  filter = (dataInputs_SameSign['standaloneDEoverE']<stacut) & (((dataInputs_SameSign['eta']>0) & ((dataInputs_SameSign['cellEdgeDeta']<-0.16) | (dataInputs_SameSign['cellEdgeDeta'] > 0.002))) | ((dataInputs_SameSign['eta']<0) & ((dataInputs_SameSign['cellEdgeDeta']<-0.002) | (dataInputs_SameSign['cellEdgeDeta'] > 0.016)))) & (((abs(dataInputs_SameSign['phi'])>1.5) & ((dataInputs_SameSign['cellEdgeDphi']<-0.001) | (dataInputs_SameSign['cellEdgeDphi'] > 0.004))) | ((abs(dataInputs_SameSign['phi'])<1.5) & ((dataInputs_SameSign['cellEdgeDphi']<-0.004) | (dataInputs_SameSign['cellEdgeDphi'] > 0.001))))
 
#(dataInputs_SameSign['cellEdgeDeta']>0.004) & (dataInputs_SameSign['cellEdgeDphi']>0.016)
  filteredData = dataInputs_SameSign[filter]
    
  filteredData.drop('cellEdgeDphi', inplace=True, axis=1)
  filteredData.drop('cellEdgeDeta', inplace=True, axis=1)
  missingOneHit_data_SameSign = filteredData.copy()

  data_SameSign_x,data_SameSign_y,data_SameSign_massess = format_data(missingOneHit_data_SameSign,allTrainingFeatures)
  data_SameSign_weights = data_SameSign_x[:,data_SameSign_x.shape[1]-1]
  data_SameSign_x = np.delete(data_SameSign_x,data_SameSign_x.shape[1]-1,1)            

  SameSign_pred = []
  for i in range(0,nFolds):
    SameSign_pred.append(bdtArray[i].predict(data_SameSign_x))
  
  print(SameSign_pred)
  counts_sameSign, binedges = np.histogram(SameSign_pred[1],bins=np.arange(0,1.01,0.04))
  print(binedges)
  print(counts_sameSign)
 

[array([2.09279788e-05, 9.35592532e-01, 2.39977944e-05, 9.85675633e-01,
       8.32054496e-01, 4.63972625e-04, 4.09901812e-04, 1.98850306e-04,
       1.24864760e-04, 1.86796224e-05, 1.46716256e-05, 1.35244656e-04,
       4.25379686e-02, 3.08132767e-05, 1.96699109e-02, 4.06051338e-01,
       3.06305592e-04, 3.76442367e-05, 1.45970853e-05, 5.29583376e-05,
       9.97484088e-01, 3.89009714e-04, 9.57250237e-01, 2.92953085e-02,
       8.21946204e-01, 3.49438924e-04, 2.11526267e-03, 2.64323771e-05,
       8.17808614e-05, 3.27728176e-03, 1.54256122e-05, 8.94519866e-01,
       1.08851245e-05, 5.38578199e-04, 1.57030125e-04, 2.19748355e-04,
       1.14664414e-04, 1.81560405e-03, 9.30641592e-01, 6.80681923e-03,
       1.26723219e-02, 6.52856543e-05, 5.63936701e-05, 3.78257653e-04,
       1.01130095e-03, 6.26335386e-04, 1.60725482e-04, 1.79021717e-05,
       1.89642105e-02], dtype=float32), array([3.95416355e-05, 7.81561852e-01, 3.65358101e-05, 9.46072400e-01,
       9.10417736e-01, 1.61369680e-0

/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


For all of the various samples, set HE depth energy values when hits don't exist to -1. So that the BDT distinguishes from hits with 0 energy

In [121]:
  sigPreds_ecalBrem = []
  stacut = -0.6
  filter = (allSignals_ecalBrem['standaloneDEoverE']<stacut) & (((allSignals_ecalBrem['eta']>0) & ((allSignals_ecalBrem['cellEdgeDeta']<-0.16) | (allSignals_ecalBrem['cellEdgeDeta'] > 0.002))) | ((allSignals_ecalBrem['eta']<0) & ((allSignals_ecalBrem['cellEdgeDeta']<-0.002) | (allSignals_ecalBrem['cellEdgeDeta'] > 0.016)))) & (((abs(allSignals_ecalBrem['phi'])>1.5) & ((allSignals_ecalBrem['cellEdgeDphi']<-0.001) | (allSignals_ecalBrem['cellEdgeDphi'] > 0.004))) | ((abs(allSignals_ecalBrem['phi'])<1.5) & ((allSignals_ecalBrem['cellEdgeDphi']<-0.004) | (allSignals_ecalBrem['cellEdgeDphi'] > 0.001))))
     
  filteredData = allSignals_ecalBrem[filter]
    
  #Find the number of close hits
  dr = 0.05
  nCloseHitsColumn = findCloseToFirstHits(filteredData,dr)
  #Add them to the dataframe
  filteredData['closeHits'] = nCloseHitsColumn
  #Apply our cut
  filteredData = filteredData[filteredData['closeHits']>1]

  signal_ecalBrem_SystWeights = filteredData.copy()
    
  filteredData.drop('cellEdgeDphi', inplace=True, axis=1)
  filteredData.drop('cellEdgeDeta', inplace=True, axis=1)
  filteredData.drop(['PUupWeight'],inplace=True, axis=1)    
  filteredData.drop(['PUdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['IDupWeight'],inplace=True, axis=1)    
  filteredData.drop(['IDdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['ISOupWeight'],inplace=True, axis=1)    
  filteredData.drop(['ISOdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['TrigUpWeight'],inplace=True, axis=1)    
  filteredData.drop(['TrigDownWeight'],inplace=True, axis=1)    
  filteredData.drop(['EnBinWeight'],inplace=True, axis=1) 
  filteredData.drop(['EventWeight_cscReweight'],inplace=True, axis=1)     
  print(filteredData)

  signal_ecalBrem_SystWeights.drop(['pt'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['eta'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['phi'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['staDR'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['staPhi'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['staE'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['staChi'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['cscDR'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_0'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_1'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_2'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_3'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_4'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_5'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['HEDepth_6'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['probeCharge'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['standaloneDEoverE'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['cscDRbyStation_0'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['cscDRbyStation_1'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['cscDRbyStation_2'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights.drop(['cscDRbyStation_3'],inplace=True, axis=1)   
  signal_ecalBrem_SystWeights.drop('cscDPhibyStation_0', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights.drop('cscDPhibyStation_1', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights.drop('cscDPhibyStation_2', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights.drop('cscDPhibyStation_3', inplace=True, axis=1)   
  signal_ecalBrem_SystWeights.drop('cscDEtabyStation_0', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights.drop('cscDEtabyStation_1', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights.drop('cscDEtabyStation_2', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights.drop('cscDEtabyStation_3', inplace=True, axis=1)       
  signal_ecalBrem_SystWeights.drop('closeHits', inplace=True, axis=1)       
    
  signal_ecalBrem_x,signal_ecalBrem_y,signal_ecalBrem_massess = format_data(filteredData,allTrainingFeatures)
  signal_ecalBrem_weights = signal_ecalBrem_x[:,signal_ecalBrem_x.shape[1]-1]
  signal_ecalBrem_x = np.delete(signal_ecalBrem_x,signal_ecalBrem_x.shape[1]-1,1)            

  signal_ecalBrem_pred = []
  for i in range(0,nFolds):
    signal_ecalBrem_pred.append(bdtArray[i].predict(signal_ecalBrem_x))
  
  print(signal_ecalBrem_pred)

  #Using predictions of testing dataset
  sig0p01_ecalBrem_preds = []
  sig0p025_ecalBrem_preds = []
  sig0p05_ecalBrem_preds = []
  sig0p1_ecalBrem_preds = []
  sig0p2_ecalBrem_preds = []
  sig0p4_ecalBrem_preds = []
  sig0p6_ecalBrem_preds = []
  sig0p8_ecalBrem_preds = []
  sig1p0_ecalBrem_preds = []
  for idx2, BDTscore in enumerate(signal_ecalBrem_pred[0]):
            if signal_ecalBrem_massess[idx2] == "0p2": sig0p2_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "0p4": sig0p4_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "0p6": sig0p6_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "0p8": sig0p8_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "1p0": sig1p0_ecalBrem_preds.append(BDTscore)
    
  print(len(sig0p2_ecalBrem_preds))
  print(len(sig0p4_ecalBrem_preds))
  print(len(sig0p6_ecalBrem_preds))
  print(len(sig0p8_ecalBrem_preds))
  print(len(sig1p0_ecalBrem_preds))
    
  sigPreds_ecalBrem.append(sig1p0_ecalBrem_preds)
  sigPreds_ecalBrem.append(sig0p2_ecalBrem_preds)
  sigPreds_ecalBrem.append(sig0p4_ecalBrem_preds)
  sigPreds_ecalBrem.append(sig0p6_ecalBrem_preds) 
  sigPreds_ecalBrem.append(sig0p8_ecalBrem_preds)


/home/krohn045/.local/lib/python3.6/site-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # This is added back by InteractiveShellApp.init_path()


             pt       eta       phi     staDR    staPhi       staE    staChi  \
0     32.837067 -1.973715  1.450745  0.160104  0.160104  26.082218  0.284379   
13    45.240620 -1.750108 -2.567751  0.025471  0.025239  36.947182  0.703174   
25    40.721760 -2.090933 -2.841632  0.140962  0.140958  22.472851  0.411775   
38    48.294486 -2.309683  0.940116  0.155033  0.150739  25.450150  0.412051   
46    49.811392  1.748761  1.405621  0.112686  0.112656  21.313950  0.533630   
...         ...       ...       ...       ...       ...        ...       ...   
9432  40.340545  2.247044 -1.640276  0.086819  0.084925  45.489586  2.010664   
9435  37.036537  2.197072  2.317092  0.380757  0.378417  12.697147  0.674359   
9437  25.859011  2.195741  0.082433  0.252056  0.246836  21.734467  1.263819   
9441  35.481769 -2.321505 -2.179989  0.040502  0.040350  39.127991  0.698950   
9452  28.901950  1.875480  2.960617  0.318684  0.312443  12.575468  0.769859   

         cscDR  HEDepth_0  HEDepth_1  .

In [122]:
  sigPreds_ecalBrem_newSamples = []
  stacut = -0.6
  filter = (allSignals_ecalBrem_newSamples['standaloneDEoverE']<stacut) & (((allSignals_ecalBrem_newSamples['eta']>0) & ((allSignals_ecalBrem_newSamples['cellEdgeDeta']<-0.16) | (allSignals_ecalBrem_newSamples['cellEdgeDeta'] > 0.002))) | ((allSignals_ecalBrem_newSamples['eta']<0) & ((allSignals_ecalBrem_newSamples['cellEdgeDeta']<-0.002) | (allSignals_ecalBrem_newSamples['cellEdgeDeta'] > 0.016)))) & (((abs(allSignals_ecalBrem_newSamples['phi'])>1.5) & ((allSignals_ecalBrem_newSamples['cellEdgeDphi']<-0.001) | (allSignals_ecalBrem_newSamples['cellEdgeDphi'] > 0.004))) | ((abs(allSignals_ecalBrem_newSamples['phi'])<1.5) & ((allSignals_ecalBrem_newSamples['cellEdgeDphi']<-0.004) | (allSignals_ecalBrem_newSamples['cellEdgeDphi'] > 0.001))))
     
  filteredData = allSignals_ecalBrem_newSamples[filter]
    
  #Find the number of close hits
  dr = 0.05
  nCloseHitsColumn = findCloseToFirstHits(filteredData,dr)
  #Add them to the dataframe
  filteredData['closeHits'] = nCloseHitsColumn
  #Apply our cut
  filteredData = filteredData[filteredData['closeHits']>1]

  signal_ecalBrem_SystWeights_newSamples = filteredData.copy()
    
  filteredData.drop('cellEdgeDphi', inplace=True, axis=1)
  filteredData.drop('cellEdgeDeta', inplace=True, axis=1)
  filteredData.drop(['PUupWeight'],inplace=True, axis=1)    
  filteredData.drop(['PUdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['IDupWeight'],inplace=True, axis=1)    
  filteredData.drop(['IDdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['ISOupWeight'],inplace=True, axis=1)    
  filteredData.drop(['ISOdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['TrigUpWeight'],inplace=True, axis=1)    
  filteredData.drop(['TrigDownWeight'],inplace=True, axis=1)    
  filteredData.drop(['EnBinWeight'],inplace=True, axis=1) 
  filteredData.drop(['EventWeight_cscReweight'],inplace=True, axis=1)     
  print(filteredData)

  signal_ecalBrem_SystWeights_newSamples.drop(['pt'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['eta'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['phi'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['staDR'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['staPhi'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['staE'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['staChi'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['cscDR'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_0'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_1'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_2'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_3'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_4'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_5'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['HEDepth_6'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['probeCharge'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['standaloneDEoverE'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['cscDRbyStation_0'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['cscDRbyStation_1'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['cscDRbyStation_2'],inplace=True, axis=1)
  signal_ecalBrem_SystWeights_newSamples.drop(['cscDRbyStation_3'],inplace=True, axis=1)   
  signal_ecalBrem_SystWeights_newSamples.drop('cscDPhibyStation_0', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights_newSamples.drop('cscDPhibyStation_1', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights_newSamples.drop('cscDPhibyStation_2', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights_newSamples.drop('cscDPhibyStation_3', inplace=True, axis=1)   
  signal_ecalBrem_SystWeights_newSamples.drop('cscDEtabyStation_0', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights_newSamples.drop('cscDEtabyStation_1', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights_newSamples.drop('cscDEtabyStation_2', inplace=True, axis=1)  
  signal_ecalBrem_SystWeights_newSamples.drop('cscDEtabyStation_3', inplace=True, axis=1)       
  signal_ecalBrem_SystWeights_newSamples.drop('closeHits', inplace=True, axis=1)       
    
  signal_ecalBrem_x,signal_ecalBrem_y,signal_ecalBrem_massess = format_data(filteredData,allTrainingFeatures)
  signal_ecalBrem_weights = signal_ecalBrem_x[:,signal_ecalBrem_x.shape[1]-1]
  signal_ecalBrem_x = np.delete(signal_ecalBrem_x,signal_ecalBrem_x.shape[1]-1,1)            

  signal_ecalBrem_pred = []
  for i in range(0,nFolds):
    signal_ecalBrem_pred.append(bdtArray[i].predict(signal_ecalBrem_x))
  
  print(signal_ecalBrem_pred)

  #Using predictions of testing dataset
  sig0p01_ecalBrem_preds = []
  sig0p025_ecalBrem_preds = []
  sig0p05_ecalBrem_preds = []
  sig0p1_ecalBrem_preds = []
  for idx2, BDTscore in enumerate(signal_ecalBrem_pred[0]):
            if signal_ecalBrem_massess[idx2] == "0p01": sig0p01_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "0p025": sig0p025_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "0p05": sig0p05_ecalBrem_preds.append(BDTscore)
            elif signal_ecalBrem_massess[idx2] == "0p1": sig0p1_ecalBrem_preds.append(BDTscore)
    

  sigPreds_ecalBrem_newSamples.append(sig0p1_ecalBrem_preds)
  sigPreds_ecalBrem_newSamples.append(sig0p05_ecalBrem_preds)
  sigPreds_ecalBrem_newSamples.append(sig0p01_ecalBrem_preds)
  sigPreds_ecalBrem_newSamples.append(sig0p025_ecalBrem_preds)

/home/krohn045/.local/lib/python3.6/site-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # This is added back by InteractiveShellApp.init_path()


              pt       eta       phi     staDR    staPhi       staE    staChi  \
4      43.041502 -1.627743 -2.173121  0.138957  0.136575  24.580366  0.454557   
21     48.733161 -1.925329  1.740154  0.040699  0.025412  46.881729  0.751696   
22     44.758526  2.086622 -0.986356  0.230913  0.226591  22.729734  0.799812   
42     48.198662 -1.761710  2.017222  0.095246  0.095227  37.296036  0.701250   
61     37.689626 -1.877131 -0.510393  0.122150  0.115927  23.542892  0.974416   
...          ...       ...       ...       ...       ...        ...       ...   
14851  37.937268 -2.331987  2.928228  0.047411  0.047370  73.481941  1.069942   
14863  43.365123  2.080743  0.724944  0.083040  0.082813  53.876198  2.246175   
14896  40.082202  2.389090 -2.737576  0.072687  0.028406  65.628098  0.428706   
14897  26.195148  1.658868 -1.148509  0.096860  0.093542  24.421108  0.809787   
14928  35.648467  1.688179  0.041659  0.060445  0.058324  31.752169  0.798001   

          cscDR  HEDepth_0 

In [123]:
  sigPreds_newSamples = []
  stacut = -0.6
  filter = (allSignals_newSamples['standaloneDEoverE']<stacut) & (((allSignals_newSamples['eta']>0) & ((allSignals_newSamples['cellEdgeDeta']<-0.16) | (allSignals_newSamples['cellEdgeDeta'] > 0.002))) | ((allSignals_newSamples['eta']<0) & ((allSignals_newSamples['cellEdgeDeta']<-0.002) | (allSignals_newSamples['cellEdgeDeta'] > 0.016)))) & (((abs(allSignals_newSamples['phi'])>1.5) & ((allSignals_newSamples['cellEdgeDphi']<-0.001) | (allSignals_newSamples['cellEdgeDphi'] > 0.004))) | ((abs(allSignals_newSamples['phi'])<1.5) & ((allSignals_newSamples['cellEdgeDphi']<-0.004) | (allSignals_newSamples['cellEdgeDphi'] > 0.001))))
     
  filteredData = allSignals_newSamples[filter]
    
  #Find the number of close hits
  dr = 0.05
  nCloseHitsColumn = findCloseToFirstHits(filteredData,dr)
  #Add them to the dataframe
  filteredData['closeHits'] = nCloseHitsColumn
  #Apply our cut
  filteredData = filteredData[filteredData['closeHits']>1]

  signal_SystWeights_newSamples = filteredData.copy()
    
  filteredData.drop('cellEdgeDphi', inplace=True, axis=1)
  filteredData.drop('cellEdgeDeta', inplace=True, axis=1)
  filteredData.drop(['PUupWeight'],inplace=True, axis=1)    
  filteredData.drop(['PUdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['IDupWeight'],inplace=True, axis=1)    
  filteredData.drop(['IDdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['ISOupWeight'],inplace=True, axis=1)    
  filteredData.drop(['ISOdownWeight'],inplace=True, axis=1)    
  filteredData.drop(['TrigUpWeight'],inplace=True, axis=1)    
  filteredData.drop(['TrigDownWeight'],inplace=True, axis=1)    
  filteredData.drop(['EnBinWeight'],inplace=True, axis=1) 
  filteredData.drop(['EventWeight_cscReweight'],inplace=True, axis=1)     
  print(filteredData)

  signal_SystWeights_newSamples.drop(['pt'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['eta'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['phi'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['staDR'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['staPhi'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['staE'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['staChi'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['cscDR'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_0'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_1'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_2'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_3'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_4'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_5'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['HEDepth_6'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['probeCharge'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['standaloneDEoverE'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['cscDRbyStation_0'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['cscDRbyStation_1'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['cscDRbyStation_2'],inplace=True, axis=1)
  signal_SystWeights_newSamples.drop(['cscDRbyStation_3'],inplace=True, axis=1)   
  signal_SystWeights_newSamples.drop('cscDPhibyStation_0', inplace=True, axis=1)  
  signal_SystWeights_newSamples.drop('cscDPhibyStation_1', inplace=True, axis=1)  
  signal_SystWeights_newSamples.drop('cscDPhibyStation_2', inplace=True, axis=1)  
  signal_SystWeights_newSamples.drop('cscDPhibyStation_3', inplace=True, axis=1)   
  signal_SystWeights_newSamples.drop('cscDEtabyStation_0', inplace=True, axis=1)  
  signal_SystWeights_newSamples.drop('cscDEtabyStation_1', inplace=True, axis=1)  
  signal_SystWeights_newSamples.drop('cscDEtabyStation_2', inplace=True, axis=1)  
  signal_SystWeights_newSamples.drop('cscDEtabyStation_3', inplace=True, axis=1)       
    
  signal_x,signal_y,signal_massess = format_data(filteredData,allTrainingFeatures)
  signal_weights = signal_x[:,signal_x.shape[1]-1]
  signal_x = np.delete(signal_x,signal_x.shape[1]-1,1)            

  signal_pred = []
  for i in range(0,nFolds):
    signal_pred.append(bdtArray[i].predict(signal_x))
  
  print(signal_ecalBrem_pred)

  #Using predictions of testing dataset
  sig0p01_preds = []
  sig0p025_preds = []
  sig0p05_preds = []
  sig0p1_preds = []
  for idx2, BDTscore in enumerate(signal_pred[0]):
            if signal_massess[idx2] == "0p01": sig0p01_preds.append(BDTscore)
            elif signal_massess[idx2] == "0p025": sig0p025_preds.append(BDTscore)
            elif signal_massess[idx2] == "0p05": sig0p05_preds.append(BDTscore)
            elif signal_massess[idx2] == "0p1": sig0p1_preds.append(BDTscore)
    

  sigPreds_newSamples.append(sig0p1_preds)
  sigPreds_newSamples.append(sig0p05_preds)
  sigPreds_newSamples.append(sig0p01_preds)
  sigPreds_newSamples.append(sig0p025_preds)

/home/krohn045/.local/lib/python3.6/site-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # This is added back by InteractiveShellApp.init_path()


              pt       eta       phi     staDR    staPhi       staE  \
4      33.209964  2.109927 -2.802178  0.202928  0.199445  22.519976   
11     38.671551  1.936469 -0.472598  0.057303  0.056059  45.571987   
18     35.329536 -2.355018 -2.647516  0.153479  0.048249  55.127998   
25     65.025207 -1.993460  2.520230  0.202763 -0.202751  33.293308   
29     36.227497 -1.974373 -0.018454  0.210331  0.205020  16.306168   
...          ...       ...       ...       ...       ...        ...   
44561  29.440846 -2.361033  1.563391  0.095344  0.092743  44.957855   
44575  47.196050  2.348442 -1.645244  0.177415  0.174963  26.222116   
44593  30.731773  1.841960 -0.985490  0.229287 -0.625876  21.488359   
44600  45.522261  2.344577  0.563520  0.147745 -0.140041  49.729023   
44625  37.722575  2.049902  1.330890  0.205786  0.204887  21.967861   

          staChi     cscDR  HEDepth_0  HEDepth_1  ...  cscDPhibyStation_2  \
4       2.387190  0.002979   0.142953   0.541251  ...           -0.004

In [125]:
outputDicts=[]
DY_BDToutput_systWeights = BDToutput_systWeights[BDToutput_systWeights['label']<0.5]
DY_BDToutput_systWeights.drop('label', inplace=True, axis=1)
DY_BDToutput_systWeights.drop('massPoint', inplace=True, axis=1)


newGenPreds_CSCqualityFiltered = []
print(len(newGenPreds[0]))
print(len(DY_BDToutput_systWeights))
print(newGenPreds[0][0])
DY_trueIndex=0
for index, row in DY_BDToutput_systWeights.iterrows():
    if row['closeHits'] > 1:
        newGenPreds_CSCqualityFiltered.append(newGenPreds[0][DY_trueIndex])
    DY_trueIndex=DY_trueIndex+1

DY_BDToutput_systWeights = DY_BDToutput_systWeights[DY_BDToutput_systWeights['closeHits']>1]        
DY_BDToutput_systWeights.drop('closeHits', inplace=True, axis=1)
        
print(len(newGenPreds[0]))
print(len(newGenPreds_CSCqualityFiltered))
print("len(sigPreds)")
print(len(sigPreds))
print(len(sigPreds[0]))
print(len(DY_BDToutput_systWeights))
idx_oldSamples = 0
idx_newSamples = 0
for idx, mass in enumerate(sigMasses):
    sigPreds_CSCqualityFiltered = []
    sigPreds_ecalBremHold = []
    print(idx)
    print(mass)
    if mass is '0p01' or mass is '0p025' or mass is '0p05' or mass is '0p1':
        print("New samples")
        sigSplitSystWeights = signal_SystWeights_newSamples[signal_SystWeights_newSamples['massPoint']==mass]
        sig_trueIndex = 0
        for index, row in sigSplitSystWeights.iterrows():
            if row['closeHits'] > 1:
                sigPreds_CSCqualityFiltered.append(sigPreds_newSamples[idx_newSamples][sig_trueIndex])
            sig_trueIndex = sig_trueIndex + 1
        sigSplitSystWeights = sigSplitSystWeights[sigSplitSystWeights['closeHits']>1]        
        sigSplitSystWeights.drop('closeHits', inplace=True, axis=1)            
        sigSplitSystWeights.drop(['label'], inplace=True, axis=1)
        sigSplitSystWeights_ecalBrem = signal_ecalBrem_SystWeights_newSamples[signal_ecalBrem_SystWeights_newSamples['massPoint']==mass]
        sigSplitSystWeights_ecalBrem.drop(['label'], inplace=True, axis=1)
        sigPreds_ecalBremHold = sigPreds_ecalBrem_newSamples[idx_newSamples]
        idx_newSamples += 1
    else:
        sigSplitSystWeights = BDToutput_systWeights[BDToutput_systWeights['massPoint']==mass]
        print(len(sigSplitSystWeights))
        sig_trueIndex = 0
        for index, row in sigSplitSystWeights.iterrows():
            if row['closeHits'] > 1:
                sigPreds_CSCqualityFiltered.append(sigPreds[idx_oldSamples][sig_trueIndex])
            sig_trueIndex = sig_trueIndex + 1
        sigSplitSystWeights = sigSplitSystWeights[sigSplitSystWeights['closeHits']>1]        
        sigSplitSystWeights.drop('closeHits', inplace=True, axis=1)            
        sigSplitSystWeights.drop(['label'], inplace=True, axis=1)
        sigSplitSystWeights_ecalBrem = signal_ecalBrem_SystWeights[signal_ecalBrem_SystWeights['massPoint']==mass]
        sigSplitSystWeights_ecalBrem.drop(['label'], inplace=True, axis=1)
        sigPreds_ecalBremHold = sigPreds_ecalBrem[idx_oldSamples]  
        idx_oldSamples += 1
    massDict = {}
    massDict['name']=mass
    massDict['mcPreds']=newGenPreds_CSCqualityFiltered
    massDict['sigPreds']=sigPreds_CSCqualityFiltered
    massDict['muPXPreds']=SameSign_pred[0]    
    massDict['sigPreds_ecalBrem']=sigPreds_ecalBremHold    
    print("len(sigPreds[idx])")
    print(len(sigPreds_CSCqualityFiltered))
    print(len(sigSplitSystWeights))
    print(len(sigPreds_ecalBremHold))
    print(len(sigSplitSystWeights_ecalBrem))
    for name, Systweights in sigSplitSystWeights.iteritems():
        massDict[name]=Systweights.values
    for name, Systweights in sigSplitSystWeights_ecalBrem.iteritems():
        massDict[name+'_ecalBrem']=Systweights.values        
    outputDicts.append(massDict)        
dyWeightsDict={}
for name,Systweights in DY_BDToutput_systWeights.iteritems():
    dyWeightsDict[name]=Systweights.values
outputDicts.append(dyWeightsDict)
pickle.dump(outputDicts, open('bdtPredictions_addedCut.pkl', 'wb'),protocol=2)

/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


101577
101577
5.206355854170397e-05
101577
93003
len(sigPreds)
5
501
93003
0
0p1
New samples


/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


len(sigPreds[idx])
3234
3234
2711
2711
1
0p05
New samples


/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


len(sigPreds[idx])
9133
9133
3524
3524
2
1p0
501
len(sigPreds[idx])
354
354
287
287
3
0p2
4035


... (4 outputs left out)

/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


len(sigPreds[idx])
2924
2924
1022
1022
7
0p6
1011


/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/home/krohn045/.local/lib/python3.6/site-packages/pandas/core/frame.py:4174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


len(sigPreds[idx])
745
745
663
663
8
0p8
713
len(sigPreds[idx])
501
501
486
486


Compared the high HCAL MC and data BDT predictions